# Batch Size: how many examples per update?

We reuse the plain MLP + Fashion-MNIST training loop (no convolutions) and change **only the batch size**.

The **batch size** is the number of training examples the network looks at before it makes *one* weight update. The loss (and so the gradient) is averaged over those examples.

- **Small batch:** many updates per epoch, each based on a noisy gradient estimate.
- **Large batch:** few updates per epoch, each based on a more accurate gradient.

**Plan**
1. Predict first, then train with batch sizes from 16 to 4096
2. Compare the *same* runs against three different x-axes: epochs, update steps, wall-clock time
3. See why a fixed learning rate is unfair to large batches, and fix it with LR scaling
4. Measure the gradient noise directly

## 0. Setup

In [ ]:
import math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Data, model, evaluation

In [ ]:
mean, std = 0.2860, 0.3530
tr = datasets.FashionMNIST('data', train=True,  download=True)
te = datasets.FashionMNIST('data', train=False, download=True)

def to_tensors(ds):
    x = (ds.data.float() / 255 - mean) / std
    return x.unsqueeze(1), ds.targets

Xtr, ytr = to_tensors(tr)
Xte, yte = to_tensors(te)

train_ds          = TensorDataset(Xtr, ytr)
train_eval_loader = DataLoader(train_ds, batch_size=1000)          # full pass, used to measure train loss
test_loader       = DataLoader(TensorDataset(Xte, yte), batch_size=1000)
N = len(train_ds)

class MLP(nn.Module):
    def __init__(self, hidden=(128, 64)):
        super().__init__()
        layers, d = [nn.Flatten()], 784
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            d = h
        layers.append(nn.Linear(d, 10))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

criterion = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total += criterion(out, y).item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total / n, correct / n

## 2. Batch size sets how many updates you get

With 60,000 training images, one epoch has `ceil(60000 / batch_size)` updates. Same number of epochs, wildly different number of updates:

In [ ]:
BATCH_SIZES = [16, 64, 256, 1024, 4096]
EPOCHS = 10
print(f"{'batch size':>10} {'steps/epoch':>12} {'steps in 10 epochs':>20}")
for b in BATCH_SIZES:
    s = math.ceil(N / b)
    print(f'{b:>10} {s:>12} {s*EPOCHS:>20}')

## 3. Training function

The same 5-step loop, wrapped in `run(batch_size, lr)`. Each run starts from the same seed (same initial weights, same data order logic). After every epoch it records the number of **steps** taken, the cumulative **training time** (evaluation time excluded), the full-dataset train loss and the test accuracy.

In [ ]:
def sync():
    if device == 'cuda':
        torch.cuda.synchronize()

def run(batch_size, lr=0.01, momentum=0.9, epochs=EPOCHS, seed=0):
    torch.manual_seed(seed)
    model = MLP().to(device)
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    h = dict(batch_size=batch_size, lr=lr, epoch=[], step=[], time=[], train_loss=[], test_acc=[], diverged=False)
    steps, elapsed = 0, 0.0
    for epoch in range(1, epochs + 1):
        sync(); t0 = time.time()
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()                 # 1. clear gradients
            loss = criterion(model(x), y)         # 2-3. forward + loss (averaged over the batch)
            loss.backward()                       # 4. backward
            optimizer.step()                      # 5. update
            steps += 1
            if not math.isfinite(loss.item()):
                h['diverged'] = True
                break
        sync(); elapsed += time.time() - t0
        if h['diverged']:
            print(f'  bs={batch_size}, lr={lr:.3g}: DIVERGED in epoch {epoch}')
            break
        trl, _ = evaluate(model, train_eval_loader)
        _, tea = evaluate(model, test_loader)
        for k, v in zip(['epoch', 'step', 'time', 'train_loss', 'test_acc'], [epoch, steps, elapsed, trl, tea]):
            h[k].append(v)
    return h

## 4. Predict first

Setting: SGD with lr 0.01 and momentum 0.9, 10 epochs, **same learning rate for every batch size**.

| Batch size | Updates in 10 epochs | Test accuracy after 10 epochs (guess) | Time per epoch vs batch 64 (faster/slower) |
|---|---|---|---|
| 16 | 37,500 | | |
| 64 | 9,380 | | |
| 256 | 2,350 | | |
| 1024 | 590 | | |
| 4096 | 150 | | |

And two more questions:
- Which batch size will have the lowest loss **after 3 epochs**?
- Which will have the lowest loss **after 1000 update steps**?

## 5. Experiment 1: same learning rate, different batch sizes

In [ ]:
exp1 = {}
for b in BATCH_SIZES:
    exp1[b] = run(b, lr=0.01)
    r = exp1[b]
    print(f'bs={b:5d}  steps={r["step"][-1]:6d}  train time={r["time"][-1]:6.1f}s  final test acc={r["test_acc"][-1]:.4f}')

### 5a. The same runs, three different x-axes

This is the most important plot of the notebook. The data are identical in all three panels; only the x-axis changes.

In [ ]:
def plot_runs(runs, title, label_fn=lambda k: f'batch {k}'):
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.2), sharey=True)
    for k, r in runs.items():
        ax[0].plot(r['epoch'], r['train_loss'], marker='o', ms=3, label=label_fn(k))
        ax[1].plot(r['step'],  r['train_loss'], marker='o', ms=3, label=label_fn(k))
        ax[2].plot(r['time'],  r['train_loss'], marker='o', ms=3, label=label_fn(k))
    ax[0].set_xlabel('epochs (passes over the data)')
    ax[1].set_xlabel('update steps'); ax[1].set_xscale('log')
    ax[2].set_xlabel('training wall-clock time (s)')
    ax[0].set_ylabel('train loss (full dataset)'); ax[0].set_yscale('log')
    ax[0].legend(); fig.suptitle(title); plt.tight_layout(); plt.show()

plot_runs(exp1, 'Same learning rate (0.01): train loss against epochs / steps / time')

How to read it:
- **Per epoch (left):** small batches win by a lot, because they make far more updates per pass over the data.
- **Per update step (middle):** the order flips. Each step with a large batch uses a better gradient, so it makes more progress, though not 256x more.
- **Per second (right):** the answer depends on the hardware. Large batches use a GPU much more efficiently, but here the data loading is done by PyTorch's `DataLoader`, which adds a per-example cost that does not shrink with the batch size, so the speed advantage of big batches is smaller than on a pure-compute workload.

### 5b. Summary table

In [ ]:
TARGET = 0.85

def summary(runs):
    rows = []
    for b, r in runs.items():
        if not r['epoch']:
            rows.append({'batch_size': b, 'verdict': 'diverged'}); continue
        hit = [i for i, a in enumerate(r['test_acc']) if a >= TARGET]
        i = hit[0] if hit else None
        rows.append({
            'batch_size': b,
            'lr': r['lr'],
            'steps/epoch': math.ceil(N / b),
            'sec/epoch': r['time'][-1] / len(r['epoch']),
            'final_train_loss': r['train_loss'][-1],
            'final_test_acc': r['test_acc'][-1],
            f'epochs_to_{int(TARGET*100)}%': r['epoch'][i] if hit else None,
            f'steps_to_{int(TARGET*100)}%':  r['step'][i]  if hit else None,
            f'sec_to_{int(TARGET*100)}%':    round(r['time'][i], 1) if hit else None,
        })
    return pd.DataFrame(rows).set_index('batch_size')

display(summary(exp1).round(4))

Note the three "to 85%" columns: the *best* batch size is a different one depending on whether you count epochs, steps or seconds. In a fixed time budget on a GPU, the best is typically a medium-to-large batch, not the smallest.

## 6. Experiment 2: the learning rate unfairly penalizes large batches

In Experiment 1, a batch of 4096 took only about 150 steps, each of the same small size (lr 0.01). It simply hadn't travelled far enough. Because a bigger batch gives a more reliable gradient, we can afford a **bigger step**. Two common rules, with batch 64 and lr 0.01 as the reference:

- **linear scaling:** `lr = 0.01 * B / 64`
- **square-root scaling:** `lr = 0.01 * sqrt(B / 64)`

Let's see how well each rule closes the gap (batch 64 is unchanged in all cases).

In [ ]:
REF_B, REF_LR = 64, 0.01
rules = {
    'fixed lr':        lambda b: REF_LR,
    'sqrt scaling':    lambda b: REF_LR * math.sqrt(b / REF_B),
    'linear scaling':  lambda b: REF_LR * b / REF_B,
}
BIG = [64, 256, 1024, 4096]

exp2 = {}
for rule, fn in rules.items():
    exp2[rule] = {}
    for b in BIG:
        exp2[rule][b] = exp1[b] if rule == 'fixed lr' else run(b, lr=fn(b))
        r = exp2[rule][b]
        acc = 'DIVERGED' if r['diverged'] else f"{r['test_acc'][-1]:.4f}"
        print(f'{rule:15s} bs={b:5d} lr={fn(b):.4f}  final test acc = {acc}')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for a, (rule, runs) in zip(ax, exp2.items()):
    for b, r in runs.items():
        a.plot(r['epoch'], r['test_acc'], marker='o', ms=3, label=f'batch {b} (lr={r["lr"]:.3g})')
    a.set_title(rule); a.set_xlabel('epoch'); a.set_ylim(0.6, 0.92); a.legend(fontsize=8)
ax[0].set_ylabel('test accuracy'); plt.tight_layout(); plt.show()

What to look for:
- **Fixed lr:** accuracy drops as the batch size grows (fewer updates).
- **Scaled lr:** the gap shrinks a lot, because each big-batch step is now bigger.
- **Linear scaling at the largest batch** uses a big lr (0.64 here). It can become unstable or noisy, since the rule only works up to a point. Real large-batch training usually adds a **warmup** (starting with a small lr), which is a good exercise.

Batch size and learning rate are not independent knobs: if you change one, you should re-tune the other.

## 7. Experiment 3: measuring gradient noise directly

A mini-batch gradient is a *noisy estimate* of the true gradient (the one computed on all 60,000 images). If the examples are independent, the noise variance should shrink like **1/B**: quadruple the batch, quarter the variance.

We check this: compute the full-data gradient once, then for each batch size draw many random mini-batches and measure how far their gradients are from the full one.

In [ ]:
def flat_grad(model, x, y):
    model.zero_grad(set_to_none=True)
    criterion(model(x), y).backward()
    return torch.cat([p.grad.flatten() for p in model.parameters()]).detach().clone()

def grad_noise(model, batch_sizes, n_samples=50, seed=123):
    model.eval()
    Xf, yf = Xtr.to(device), ytr.to(device)
    g_full = flat_grad(model, Xf, yf)
    gen = torch.Generator(device='cpu').manual_seed(seed)
    out = {}
    for b in batch_sizes:
        errs = []
        for _ in range(n_samples):
            idx = torch.randperm(N, generator=gen)[:b].to(device)
            errs.append((flat_grad(model, Xf[idx], yf[idx]) - g_full).pow(2).sum().item())
        out[b] = float(np.mean(errs))
    return out

NOISE_BS = [4, 16, 64, 256, 1024, 4096]

# gradient noise at initialization
torch.manual_seed(0)
noise_init = grad_noise(MLP().to(device), NOISE_BS)

# ...and after 3 epochs of ordinary training (batch 128), to compare
torch.manual_seed(0)
m = MLP().to(device)
opt = optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
loader = DataLoader(train_ds, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(0))
for _ in range(3):
    m.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); criterion(m(x), y).backward(); opt.step()
noise_trained = grad_noise(m, NOISE_BS)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name, d in [('at initialization', noise_init), ('after 3 epochs', noise_trained)]:
    ax.loglog(list(d), list(d.values()), marker='o', label=name)
b = np.array(NOISE_BS, dtype=float)
ax.loglog(b, noise_init[NOISE_BS[0]] * NOISE_BS[0] / b, 'k--', alpha=.5, label='1 / B reference line')
ax.set_xlabel('batch size B'); ax.set_ylabel('E || g_batch - g_full ||^2  (gradient noise)')
ax.set_title('Mini-batch gradient noise vs batch size'); ax.legend(); plt.show()

The points follow a line of slope -1 on the log-log plot: quadruple the batch size and the noise variance falls to a quarter. That is why bigger batches give "smoother" training, and why the benefit of ever-larger batches has diminishing returns for *steps*: halving the noise needs double the batch, i.e. double the compute per step.

## Takeaways

1. **Batch size = examples averaged per update.** The number of updates per epoch is `N / batch_size`.
2. **Small batch:** many noisy updates. Better progress per epoch, but poor use of the hardware.
3. **Large batch:** few accurate updates. Fast per epoch on a GPU, but at the same learning rate it learns less per epoch.
4. **Compare fairly:** look at the same runs per epoch, per step, *and* per second, since the winner changes.
5. **Tune lr together with batch size.** Scale the learning rate up with the batch (linear or sqrt rule, with warmup for very large batches).
6. Gradient noise falls as **1/B**, so bigger and bigger batches buy less and less.

## Exercises
1. Add a **warmup** (ramp lr from 0 to its target over the first epoch) to the linear-scaled batch-4096 run. Does it become stable?
2. Repeat the fixed-lr sweep with `optim.Adam(lr=1e-3)`. Is the batch-size sensitivity different?
3. Run batch 16 and batch 4096 with 3 different seeds. Which one varies more from seed to seed?
4. Re-do the timing with `num_workers=2` in the `DataLoader`. How do the seconds-per-epoch change?
5. Train batch 4096 for 160 epochs (the same number of *updates* as batch 64 for 10 epochs). Does it catch up?
6. Add a column to the summary table: `final_test_acc - final_train_acc`. Do small batches overfit less?